# TSTR study: sample the new synthetic sets

Generates the synthetic sets the full TSTR study needs (Amendment 1 of the study design, fixed on
Oct 27 before any of these sets existed): the three spectrogram paths at two checkpoints and two
guidance values.

| Path | Model | Checkpoints | Guidance | Sets |
|---|---|---|---|---|
| A-64 | Path A, 64 mel bands | EMA at 5k and 10k steps | 1.0 and 3.0 | 4 |
| A-128 | Path A, 128 mel bands | EMA at 5k and 10k steps | 1.0 and 3.0 | 4 |
| C | Path C, 64 linear bands | EMA at 5k and 10k steps | 1.0 and 3.0 | 4 |

The three sets at 10k steps and guidance 1.0 already exist (they are the sets every earlier result
used), so **9 new sets** are sampled. Path B keeps its three existing sets.

Every set: 800 clips, 16 per class, DDIM with the training config's step count, saved with the
checkpoint, guidance and code version. Nothing is scored here.

**Resumable.** A set whose file already exists is skipped, so the notebook can be rerun after a
Colab disconnect or GPU limit and continues where it stopped. Runtime on a T4: roughly 5-10 minutes
per set at guidance 1.0 and twice that at guidance 3.0 (each step then runs the model twice), so
about 1.5-2 hours for all 9. It can be split over several sessions.

## 1. Setup

In [ ]:
!pip install -q diffusers accelerate soundfile
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, time, subprocess
REPO_DIR = "/content/thesis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/1jvaldez1/audio-diffusion-synth.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
sys.path.insert(0, REPO_DIR)
import common as C

import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from diffusers.training_utils import EMAModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
commit = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"]).decode().strip()

## 2. Config

`RUN_DIRS` are the training runs; each holds `ema_step005000.pt` and `ema_step010000.pt` (EMA
weights saved during training). If a 10k milestone is missing, the EMA weights in `last.pt` are
used instead, which are the same weights.

In [ ]:
QUICK_TEST = False          # True: 1 clip per class, 5 sampling steps, a separate folder

cfg64, cfg128 = C.Config(), C.Config(n_mels=128)
ROOT = cfg64.out_root
OUT = Path(f"{ROOT}/samples/study" + ("_quick" if QUICK_TEST else ""))
OUT.mkdir(parents=True, exist_ok=True)

PATHS = {                     # name -> (run dir, config, representation)
    "A64":  (f"{ROOT}/path_a",        cfg64,  "mel"),
    "A128": (f"{ROOT}/path_a_128mel", cfg128, "mel"),
    "C":    (f"{ROOT}/path_c",        cfg64,  "linear"),
}
EXISTING = {                  # the sets at 10k / guidance 1.0 that already exist: not resampled
    "A64":  f"{ROOT}/samples/path_a.pt",
    "A128": f"{ROOT}/samples/path_a_128mel.pt",
    "C":    f"{ROOT}/samples/path_c_linear.pt",
}
CHECKPOINTS = (5000, 10000)
GUIDANCE = (1.0, 3.0)
N_PER_CLASS = 1 if QUICK_TEST else 16
DDIM_STEPS = 5 if QUICK_TEST else None          # None: the training config's inference_steps
BATCH = 64                                      # lower if the GPU runs out of memory

def set_file(path, step, g):
    return OUT / f"{path}_ema{step // 1000}k_g{g:g}.pt"

todo = [(p, s, g) for p in PATHS for s in CHECKPOINTS for g in GUIDANCE
        if not (s == 10000 and g == 1.0) and not set_file(p, s, g).exists()]
done = [(p, s, g) for p in PATHS for s in CHECKPOINTS for g in GUIDANCE
        if not (s == 10000 and g == 1.0) and set_file(p, s, g).exists()]
print(f"to sample: {len(todo)} | already saved: {len(done)}")
for p, s, g in todo:
    print(f"   {set_file(p, s, g).name}")

## 3. Load a checkpoint

Rebuilds each path's UNet exactly as its training notebook did and copies in the EMA weights. The
input layer is read from the checkpoint, so a run trained with Path A's optional frequency channel
gets it back. A run is refused if its config does not match the one used here (band count, clip
length, training length).

In [ ]:
def load_ema_model(run_dir, cfg, step):
    run_dir = Path(run_dir)
    f = run_dir / f"ema_step{step:06d}.pt"
    if f.exists():
        ck = torch.load(f, map_location="cpu")
    elif step == cfg.total_steps and (run_dir / "last.pt").exists():
        print(f"   {f.name} not found; using the EMA weights in last.pt (the same weights)")
        ck = torch.load(run_dir / "last.pt", map_location="cpu")
        assert ck["step"] == cfg.total_steps, f"last.pt is at step {ck['step']}, not {cfg.total_steps}"
    else:
        raise FileNotFoundError(f"no EMA weights for step {step} in {run_dir}")
    rc = ck["config"]
    for k in ("n_mels", "spec_frames", "hop_length", "n_fft", "sample_rate", "total_steps", "num_classes"):
        assert rc.get(k) == getattr(cfg, k), f"{run_dir.name}: {k} = {rc.get(k)} in the checkpoint, {getattr(cfg, k)} here"

    probe = C.build_unet2d(cfg, in_channels=1)
    names = [n for n, _ in probe.named_parameters()]
    in_ch = ck["ema"]["shadow_params"][names.index("conv_in.weight")].shape[1]
    unet = C.build_unet2d(cfg, in_channels=in_ch).to(device)
    ema = EMAModel(unet.parameters(), decay=0.999)
    ema.load_state_dict(ck["ema"]); ema.copy_to(unet.parameters()); unet.eval()

    if in_ch == 2:        # Path A's frequency-position channel: a fixed ramp from -1 to +1
        ramp = torch.linspace(-1, 1, cfg.n_mels, device=device).view(1, 1, cfg.n_mels, 1)
        def forward_fn(model, x, t, labels):
            r = ramp.expand(x.shape[0], 1, cfg.n_mels, x.shape[-1]).to(x.dtype)
            return model(torch.cat([x, r], dim=1), t, class_labels=labels).sample
    else:
        def forward_fn(model, x, t, labels):
            return model(x, t, class_labels=labels).sample
    return unet, forward_fn, in_ch, sum(p.numel() for p in unet.parameters())

## 4. Sample

One set at a time, saved as soon as it is done. Each set has its own fixed seed (derived from its
name), so a rerun reproduces it.

In [ ]:
import zlib
t_all = time.time()
for path, step, g in todo:
    run_dir, cfg, rep = PATHS[path]
    out = set_file(path, step, g)
    if out.exists():
        continue
    print(f"{out.name}: loading {Path(run_dir).name} at step {step}")
    unet, fwd, in_ch, n_params = load_ema_model(run_dir, cfg, step)
    labels = C.balanced_labels(cfg, N_PER_CLASS)
    C.seed_everything(zlib.crc32(out.stem.encode()) % 2**31)
    t0 = time.time()
    x = C.sample(unet, fwd, (1, cfg.n_mels, cfg.spec_frames), cfg, device, labels=labels,
                 guidance=g, steps=DDIM_STEPS, batch_size=BATCH)
    meta = dict(path=path, guidance=g, ema_step=step, steps=DDIM_STEPS or cfg.inference_steps, n_params=n_params,
                train_steps=cfg.total_steps, use_freq_channel=in_ch == 2, commit=commit, study="tstr_full", quick=QUICK_TEST)
    C.save_samples(out, x, labels, cfg, representation=rep, **meta)
    print(f"   saved {tuple(x.shape)} in {(time.time() - t0) / 60:.1f} min | range [{x.min():.2f}, {x.max():.2f}]")
    del unet; torch.cuda.empty_cache() if device == "cuda" else None
print(f"\nall done in {(time.time() - t_all) / 60:.1f} min")

## 5. Check

Every set of the study, new and existing, with its clip count and value range, and one spectrogram
per set for the same class. Guidance 3.0 should look sharper and less varied than 1.0; the 5k
checkpoint noisier than 10k.

In [ ]:
grid = {p: {(s, g): (set_file(p, s, g) if not (s == 10000 and g == 1.0) else Path(EXISTING[p]))
            for s in CHECKPOINTS for g in GUIDANCE} for p in PATHS}
cls = 0
fig, axes = plt.subplots(len(PATHS), 4, figsize=(13, 2.6 * len(PATHS)), squeeze=False)
for i, p in enumerate(PATHS):
    for j, ((s, g), f) in enumerate(grid[p].items()):
        ax = axes[i, j]
        if not f.exists():
            ax.set_title(f"{p} {s // 1000}k g{g:g}: missing", fontsize=8); ax.axis("off"); continue
        d = C.load_samples(f)
        x, y = d["samples"], d["labels"]
        print(f"{p:<5} {s // 1000:>2}k g{g:g}  {tuple(x.shape)}  per class {torch.bincount(y).min().item()}-{torch.bincount(y).max().item()}"
              f"  range [{x.min():.2f}, {x.max():.2f}]  {f.name}")
        ax.imshow(x[y == cls][0, 0], origin="lower", aspect="auto", cmap="magma", vmin=-1, vmax=1)
        ax.set_title(f"{p} {s // 1000}k g{g:g}", fontsize=8); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()